In [ ]:
%matplotlib tk 

import numpy as np
from numpy.random import rand
from landscapeWithOcean import LandscapeWithOcean

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from matplotlib import cm

In [ ]:
def AddHill(Z,NX,NY,xx,yy,r,h):

    for x in range(NX):
        for y in range(NY):
            dx = np.mod(x-xx+NX/2,NX)-NX/2; 
            dy = np.mod(y-yy+NY/2,NY)-NY/2;
            dr = np.sqrt(dx**2+dy**2);
            if (dr<r):
                Z[x,y] += h * (np.cos(dr/r*np.pi/2.0))**2;

    return Z

In [ ]:
# ### Define simulation grid and initial conditions

# NX = 70#*2 #number of rows
# NY = 70#*2 #number of columns

# d  = 5 # grid spacing in meters
# dx = d # keep dx=dy for simplicity
# dy = d

# LX=NX*dx
# LY=NY*dy

# # small random features in topography to begin erosion
# Z = rand(NX,NY)*0.1
# for i in range(5):
#     xx = rand()*NX
#     yy = rand()*NY
#     r  = (0.1+rand())*NX
#     h  = (0.1+rand())*5
#     Z = AddHill(Z,NX,NY,xx,yy,r,h)

# for i in range(5):
#     xx = rand()*NX
#     yy = rand()*NY
#     r  = (0.1+rand())*NX/2
#     h  = (0.5+rand())*10
#     Z = AddHill(Z,NX,NY,xx,yy,r,h)

# x = np.arange(NX)
# y = np.arange(NY)
# X,Y = np.meshgrid(y,x) #strange that y goes first !!!
# ZMaxOrg = np.max(Z)
# #print(ZMaxOrg)

In [ ]:

NX = 140
NY = 140

d = 5
dx = d
dy = d

LX = NX * dx
LY = NY * dy

x = np.linspace(-NX/2, NX/2, NX)
y = np.linspace(-NY/2, NY/2, NY)
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)

R_volcano = 25
R_caldera = 10
height_volcano = 20
depth_caldera = 12
outer_slope = 0.3

Z = np.zeros((NX, NY))

Z += height_volcano * np.exp(-((R / R_volcano)**2))

Z -= depth_caldera * np.exp(-((R / R_caldera)**2) * 2)

Z += outer_slope * (R / np.max(R))

Z += 0.05 * X
Z += 0.02 * Y

Z -= np.min(Z)
Z /= np.max(Z)
Z *= 20.0
Z += 0.5

x = np.arange(NX)
y = np.arange(NY)
X, Y = np.meshgrid(y, x)
ZMaxOrg = np.max(Z)


In [ ]:
### Physical Parameters
K = 5e-6 # meters^(1-2m)/yr

D = 0.01 # m^2/yr

# uplift rate
# uplift = 0.03 / 600.
uplift = 0.0

In [ ]:
### Model parameters

# Time step
dt = 300 #d**2 / D / 8. 
#dt = d**2 / D /16
print(' dt[years] = ',dt)

#Area exponent A^m, default m=1
m=0.7

#gradient exponent g^n, default n=1
n=1.7

#erosion threshold 
theta_c = 2 

In [ ]:
# Total simulation time
T = 1000.0 * 625.0

# total number of iterations
n_iter = int(np.round(T/dt))
print('Number of interation: ',n_iter)

In [ ]:
ls = LandscapeWithOcean(NX,NY)

oceanLevelParameter=0.2
ls.ComputeOceanVolumeFromOceanLevelParameter(Z,NX,NY,oceanLevelParameter)

ls.pool_check(Z,NX,NY)
ls.A = np.zeros((NX,NY))

In [ ]:
# Set-up figure
def init_figure():
    fig = plt.figure(figsize=(12.,6.))
    plt.show()
    return fig

def update_figure():
        plt.clf()
        ax1 = fig.add_subplot(121,projection='3d')

        # use equal x-y aspect with an explicit vertical exageration
        vert_exag = 4.
        ax1.set_xlim3d(0,max(NX,NY))
        ax1.set_ylim3d(0,max(NX,NY))
        ax1.set_zlim3d(0,ZMaxOrg)

        ax1.set_title('Surface Relief')


        ZPlot = np.copy(Z)
        ZPlot[ZPlot<ls.ZBeachLevel] = ls.ZBeachLevel 
        ZPlot -= ls.ZBeachLevel
        ax1.plot_surface(X,Y,ZPlot, cmap = cm.terrain, rstride=1, cstride=1,
                antialiased=False,linewidth=0)

        ax2 = fig.add_subplot(122,aspect='equal')
        ax2.set_title('Elevation')

        #im = ax2.pcolor(Z,cmap=cm.terrain)
        im = ax2.pcolor(ZPlot,cmap=cm.coolwarm)
        cs = ax2.contour(ZPlot,6,colors='k')

        # Add a color bar which maps values to colors.
        cbar = fig.colorbar(im, shrink=0.5, aspect=5)
        # Add the contour line levels to the colorbar
        cbar.add_lines(cs)

        #plt.show()
        plt.draw()
        plt.pause(0.0001)

In [ ]:
from matplotlib.animation import FFMpegWriter

metadata = dict(title='Landscape Evolution', artist='Matplotlib', comment='Volcano caldera erosion')
writer = FFMpegWriter(fps=10, metadata=metadata, bitrate=180000)

fig = init_figure()
update_figure()
Znew = np.copy(Z)

with writer.saving(fig, "volcano_evolution2.mp4", dpi=150):

    for it in range(1, n_iter+1):
        ls.calculate_collection_area(Z, NX, NY)
        ls.A *= dx * dy

        for i in range(NX):
            iL = np.mod(i-1, NX)
            iR = np.mod(i+1, NX)
            for j in range(NY):
                jD = np.mod(j-1, NY)
                jU = np.mod(j+1, NY)

                if ls.ocean[i,j] > 0:
                    Psi_z = 0
                    Phi_z = 0
                else:
                    if ls.drain[i,j] > 0:
                        s1 = (Z[iR,j]  - Z[iL,j]) / (2.*dx)
                        s2 = (Z[i,jU]  - Z[i,jD]) / (2.*dy)
                        s3 = (Z[iR,jD] - Z[iL,jU]) / (2. * np.sqrt(dx**2 + dy**2))
                        s4 = (Z[iR,jU] - Z[iL,jD]) / (2. * np.sqrt(dx**2 + dy**2))
                        gradient = (np.sqrt(s1**2 + s2**2) + np.sqrt(s3**2 + s4**2)) / 2.
                        Psi_z = K * (ls.A[i,j]**m * gradient**n - theta_c)

                    elif ls.drainage[i,j] > 0:
                        if (Z[i,j] >= Z[iR,j]) and ls.pool[iR,j] != ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[iR,j]) / dx
                        elif (Z[i,j] >= Z[iL,j]) and ls.pool[iL,j] != ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[iL,j]) / dx
                        elif (Z[i,j] >= Z[i,jU]) and ls.pool[i,jU] != ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[i,jU]) / dy
                        elif (Z[i,j] >= Z[i,jD]) and ls.pool[i,jD] != ls.drainage[i,j]:
                            gradient = (Z[i,j]-Z[i,jD]) / dy
                        else:
                            gradient = 0.02
                        Psi_z = K * (ls.A[i,j]**m * gradient**n - theta_c)

                    else:
                        Psi_z = 0

                    if (Psi_z < 0):
                        Psi_z = 0

                    Phi_z = D * ((Z[iR,j] - 2.*Z[i,j] + Z[iL,j]) / dx**2 +
                                 (Z[i,jU] - 2.*Z[i,j] + Z[i,jD]) / dy**2)

                Znew[i,j] = Z[i,j] + (Phi_z - Psi_z + uplift) * dt

                dZdt = (Znew[i,j] - Z[i,j]) / dt
                CFL = abs(dZdt) * dt / min(dx, dy)
                if (CFL > 1.0):
                    print('\nWarning: Time step of', dt, 'is probably too large. Safer would be:', dt/CFL)

                if (Znew[i,j] < 0.):
                    Znew[i,j] = 0.

        Z = np.copy(Znew)
        ls.pool_check(Z, NX, NY)

        if (np.mod(it, 10) == 0):
            print(it, end='')
            update_figure()
            writer.grab_frame()   # <<< capture the current figure frame
            print(' Ocean level=', ls.ZBeachLevel, ' Ocean surface fraction=', 100*ls.AOcean/(NX*NY))
        else:
            print('.', end='')

    update_figure()
    writer.grab_frame()

print('Simulation finished and saved as volcano_evolution.mp4')
